# Chapter 13 — Problem Set 2: Solutions

Reference implementations for the advanced problem set.

---
*Generated by Berta AI*

In [ ]:
import sys, os, re, time, json
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CORPUS_PATH = os.path.join('..', '..', 'datasets', 'sample_corpus.txt')
with open(CORPUS_PATH) as f:
    raw = f.read()
pattern = re.compile(r'^\[(doc-\d+)\]\s*(.+)$', re.MULTILINE | re.DOTALL)
documents = {}
for para in re.split(r'\n\s*\n', raw):
    m = pattern.match(para.strip())
    if m:
        documents[m.group(1)] = m.group(2).strip()

ids = list(documents.keys())
texts = list(documents.values())
queries_df = pd.read_csv(os.path.join('..', '..', 'datasets', 'queries.csv'))
queries_df['relevant'] = queries_df['relevant_doc_ids'].str.split('|')

from rag_pipeline import TfidfEmbedder, RAGPipeline, MockLLM
from chunking import Chunker
from vectorstore import InMemoryVectorStore, BM25Index, HybridIndex, reciprocal_rank_fusion

print('Loaded:', len(documents), 'docs;', len(queries_df), 'queries')

## Problem 1 — Hybrid retrieval

In [ ]:
embedder = TfidfEmbedder(dim=128).fit(texts)
embs = embedder.encode(texts)

dense = InMemoryVectorStore(dim=embs.shape[1])
dense.add(embeddings=embs, chunk_ids=ids, texts=texts)

sparse = BM25Index()
sparse.add(chunk_ids=ids, texts=texts)

hybrid = HybridIndex(dense=dense, sparse=sparse, rrf_k=60)

def hit_at_k(retrieved_ids, gold, k):
    return int(any(r in set(gold) for r in retrieved_ids[:k]))

scores = {'dense': 0, 'sparse': 0, 'hybrid': 0}
for _, row in queries_df.iterrows():
    qe = embedder.encode_query(row['query'])
    d = [r.chunk_id for r in dense.search(qe, top_k=3)]
    s = [r.chunk_id for r in sparse.search(row['query'], top_k=3)]
    h = [r.chunk_id for r in hybrid.search(row['query'], qe, top_k=3)]
    scores['dense']  += hit_at_k(d, row['relevant'], 3)
    scores['sparse'] += hit_at_k(s, row['relevant'], 3)
    scores['hybrid'] += hit_at_k(h, row['relevant'], 3)

n = len(queries_df)
for k, v in scores.items():
    print(f'{k:6} hit@3 = {v}/{n} = {v/n:.2f}')

## Problem 2 — Query rewriting

In [ ]:
SYNONYMS = {
    'rag': 'retrieval-augmented generation',
    'bm25': 'bm25 sparse retriever term frequency',
    'cosine': 'cosine similarity dot product normalized',
    'hyde': 'hyde hypothetical document embedding',
}

def expand_with_synonyms(q):
    out = q
    for k, v in SYNONYMS.items():
        if re.search(rf'\b{k}\b', q, flags=re.I):
            out += ' ' + v
    return out

def multi_query(q, n=3):
    base = q.split()
    variants = [q]
    for i in range(1, n):
        if len(base) > 3:
            variants.append(' '.join(base[i:] + base[:i]))
    return variants

for q in queries_df['query'].head(3):
    print('orig    :', q)
    print('expanded:', expand_with_synonyms(q))
    print('variants:', multi_query(q))
    print()

## Problem 3 — Faithfulness scorer

In [ ]:
def faithfulness(answer, contexts):
    sents = [s.strip() for s in re.split(r'[.!?]', answer) if s.strip()]
    if not sents:
        return 0.0
    ctx_words = set()
    for c in contexts:
        text = c.text if hasattr(c, 'text') else c
        ctx_words |= {w.lower() for w in text.split() if len(w) > 3}
    supported = sum(
        1 for s in sents
        if len({w.lower() for w in s.split() if len(w) > 3} & ctx_words) >= 2
    )
    return supported / len(sents)

# Tests
ctx = [type('C', (), {'text': 'BM25 is a sparse retriever using term frequency.'})()]
print(faithfulness('BM25 uses term frequency.', ctx))                # ~1.0
print(faithfulness('BM25 uses term frequency. Also, dragons.', ctx)) # ~0.5
print(faithfulness('Dragons rule the kingdom of vectors.', ctx))     # 0.0

## Problem 4 — Multi-hop retrieval

In [ ]:
pipe = RAGPipeline(
    chunker=Chunker(strategy='sentence', max_tokens=80),
    embedder=embedder,
    llm_client=MockLLM(),
)
pipe.index_documents(documents)

question = "Compare BM25 to FAISS for production retrieval."

hop1 = pipe.retrieve(question, top_k=3)
seed = hop1[0].text.split('.')[0]   # last-mile heuristic
hop2 = pipe.retrieve(seed, top_k=3)

union = {h.chunk_id: h for h in (hop1 + hop2)}
print('hop1:', [h.chunk_id for h in hop1])
print('hop2:', [h.chunk_id for h in hop2])
print('union size:', len(union))

## Problem 5 — Evaluation harness

In [ ]:
def chunk_to_doc(cid):
    """Strip chunker suffix (e.g. 'doc-001::sent::0' -> 'doc-001')."""
    return cid.split('::', 1)[0]

def evaluate(retrieve_fn, queries_df, top_ks=(1, 3, 5), id_map=chunk_to_doc):
    rows = []
    macro = {k: 0 for k in top_ks}
    for _, row in queries_df.iterrows():
        rids = [id_map(h.chunk_id) for h in retrieve_fn(row['query'], top_k=max(top_ks))]
        record = {'query': row['query'][:50]}
        for k in top_ks:
            hit = int(any(r in set(row['relevant']) for r in rids[:k]))
            record[f'hit@{k}'] = hit
            macro[k] += hit
        rows.append(record)
    df = pd.DataFrame(rows)
    print('Macro hit@k:', {k: round(v / len(queries_df), 2) for k, v in macro.items()})
    return df

# Use the pipeline's retriever
df = evaluate(pipe.retrieve, queries_df)
df.head()

## Problem 6 — Latency profiling

In [ ]:
def time_it(fn, *a, **kw):
    t0 = time.perf_counter()
    out = fn(*a, **kw)
    return out, (time.perf_counter() - t0) * 1000

q = "How does hybrid search combine dense and sparse retrieval?"
emb, t_embed   = time_it(embedder.encode_query, q)
_,   t_dense   = time_it(dense.search, emb, 5)
_,   t_sparse  = time_it(sparse.search, q, 5)
_,   t_hybrid  = time_it(hybrid.search, q, emb, 5)
_,   t_llm     = time_it(MockLLM().complete, "Question: x\nContext: y\nAnswer:")

stages = ['embed_q', 'dense', 'sparse', 'rrf+hybrid', 'mock_llm']
times  = [t_embed, t_dense, t_sparse, t_hybrid, t_llm]
print(dict(zip(stages, [round(t, 2) for t in times])))

plt.figure(figsize=(7, 3))
plt.barh(stages, times, color='steelblue')
plt.xlabel('latency (ms)')
plt.title('RAG stage latency (CPU, mock LLM)')
plt.tight_layout()
plt.show()

---
*Generated by Berta AI*